# BÔNUS LLM — Parte 2: LLMOps (observabilidade, qualidade de resposta e drift)

No bônus anterior, o MLflow mostrou a **acurácia** de cada modelo. Para um LLM isso não basta:
a resposta é um texto, não existe "resposta certa" única e o custo depende de cada chamada.

| Parte | Pergunta | Recurso do MLflow |
|---|---|---|
| **A. Tracing** | O que aconteceu em cada requisição? Quanto custou? | Traces (spans, tokens, latência) |
| **B. Qualidade** | As respostas são boas? O prompt v2 é melhor que o v1? | `mlflow.genai.evaluate` + scorers |
| **C. Drift** | O comportamento mudou com o tempo? | `mlflow.search_traces` + runs de monitoramento |

> Pré-requisito: ter executado o notebook `3_llm_pipeline.ipynb` (ele registra o prompt `mentoria` v1).

In [ ]:
import os, re, time, json, requests
import numpy as np
import pandas as pd
import mlflow
from openai import OpenAI

MLFLOW_URI = os.environ["MLFLOW_TRACKING_URI"]
MENTORIA_URL = os.environ["MENTORIA_URL"]
CLASSIFICADOR_URL = os.environ["CLASSIFICADOR_URL"]
LLM_MODELO = os.environ["LLM_MODELO"]
EMBED_MODELO = os.environ["EMBED_MODELO"]

mlflow.set_tracking_uri(MLFLOW_URI)
llm = OpenAI()
EXP_MENTORIA = mlflow.set_experiment("LLM Mentoria").experiment_id   # onde o serviço grava os traces

FEATURES = ["REPROVACOES_MAT_1", "REPROVACOES_MAT_2", "REPROVACOES_MAT_3", "REPROVACOES_MAT_4",
            "NOTA_MAT_1", "NOTA_MAT_2", "NOTA_MAT_3", "NOTA_MAT_4",
            "INGLES", "H_AULA_PRES", "TAREFAS_ONLINE", "FALTAS"]
alunos = pd.read_csv("avaliacao_mentoria.csv")

def aluno_json(linha):
    return {k: (v.item() if hasattr(v, "item") else v) for k, v in linha.items() if k != "PERFIL_REAL"}

def mentoria(aluno, **opcoes):
    return requests.post(f"{MENTORIA_URL}/mentoria", json={"aluno": aluno, **opcoes}, timeout=600).json()

IP = requests.get("http://checkip.amazonaws.com").text.strip()
print("MLflow UI: http://%s:8089  -> experimento 'LLM Mentoria' -> aba Traces" % IP)

---
# Parte A — Tracing: observando cada requisição

O serviço `mentoria` registra um **trace** por requisição, com um **span** para cada etapa:

```
mentoria (CHAIN)
 ├── classificar (TOOL)            -> chamada HTTP ao classificador da Etapa 2
 └── Completions (CHAT_MODEL)      -> chamada ao LLM: prompt, resposta, tokens (mlflow.openai.autolog)
```

In [ ]:
for _, linha in alunos.head(3).iterrows():
    r = mentoria(aluno_json(linha), janela="parte_a")
    print(f"{r['perfil']:<12} {r['latencia_llm_s']:5.1f}s  {r['tokens_saida']} tokens  trace={r['trace_id']}")

**No MLflow:** Experiments → `LLM Mentoria` → aba **Traces**. Abra um trace e responda:

> - Quais spans aparecem e quanto tempo cada um levou? Qual etapa domina a latência?
> - O span do LLM mostra o prompt **exato** enviado. O nome do aluno aparece em algum lugar do trace? Por quê?
> - Quais tags o serviço gravou (perfil, modelo, prompt_versao, janela)? Para que elas servem?

In [ ]:
def traces_df(*janelas, max_results=1000):
    """Traces do serviço como tabela: uma linha por requisição (filtra pelas janelas pedidas)."""
    df = mlflow.search_traces(locations=[EXP_MENTORIA], max_results=max_results)
    df = df[df["tags"].apply(lambda t: t.get("janela") in janelas)]
    linhas = []
    for _, t in df.iterrows():
        uso = json.loads(t["trace_metadata"].get("mlflow.trace.tokenUsage", "{}"))
        req = t["request"] if isinstance(t["request"], dict) else json.loads(t["request"] or "{}")
        resp = t["response"] if isinstance(t["response"], dict) else json.loads(t["response"] or "{}")
        linhas.append({
            "trace_id": t["trace_id"], "quando": t["request_time"],
            "latencia_s": t["execution_duration"] / 1000,
            "tokens_entrada": uso.get("input_tokens"), "tokens_saida": uso.get("output_tokens"),
            **{k: t["tags"].get(k) for k in ["janela", "perfil", "modelo", "prompt_versao", "raciocinio"]},
            "texto": (resp or {}).get("recomendacao", ""),
            **{f: (req.get("aluno") or {}).get(f) for f in FEATURES},
        })
    return pd.DataFrame(linhas)

time.sleep(3)   # o serviço envia os traces de forma assíncrona
df_a = traces_df("parte_a")
df_a[["perfil", "latencia_s", "tokens_entrada", "tokens_saida", "modelo", "prompt_versao"]]

### Custo: FinOps de LLM

O Ollama não cobra, mas em produção cada token tem preço. Usando uma **tabela de preços ilustrativa** (US$ por 1 milhão de tokens):

In [ ]:
PRECO = {"entrada": 0.15, "saida": 0.60}   # valores ilustrativos de um modelo pequeno comercial (US$ / 1M tokens)

df_a["custo_usd"] = (df_a.tokens_entrada * PRECO["entrada"] + df_a.tokens_saida * PRECO["saida"]) / 1e6
custo_medio = df_a.custo_usd.mean()
print(f"Custo médio por recomendação: US$ {custo_medio:.6f}")
print(f"20 mil alunos x 4 recomendações/semestre: US$ {custo_medio * 20000 * 4:,.2f}")

### O custo escondido do "raciocínio"

Modelos como o `qwen3` podem **pensar antes de responder** (tokens de raciocínio). Isso pode melhorar a resposta, mas consome tokens e tempo.
O serviço desliga o raciocínio por padrão. Compare (esta célula demora ~1 min):

In [ ]:
aluno = aluno_json(alunos.iloc[0])
for raciocinio in [False, True]:
    t = time.time()
    r = mentoria(aluno, raciocinio=raciocinio, janela="parte_a_raciocinio")
    print(f"raciocinio={raciocinio!s:<5} | {time.time() - t:5.1f}s | tokens de saída: {r['tokens_saida']} | "
          f"texto: {len(r['recomendacao'])} caracteres")

> **Responda:** com raciocínio ligado, o texto final ficou maior ou menor? Para onde foram os tokens?
> O que acontece se o limite de tokens (`max_tokens`) acabar durante o raciocínio? (veja o trace no MLflow)

---
# Parte B — Response quality: avaliando as respostas

Acurácia não se aplica a texto. Em LLMOps usamos **avaliadores (scorers)**:

| Tipo | Exemplo | Custo |
|---|---|---|
| **Heurística (código)** | o texto cita o perfil? tem dado pessoal? | quase zero |
| **LLM-as-a-judge** | um LLM dá nota à resposta de outro | caro e lento |
| **Avaliação humana** | o coordenador aprova ou não | muito caro |

Vamos avaliar o prompt **v1** com 4 scorers de código, criar um prompt **v2** e comparar, **antes** de colocá-lo em produção.

In [ ]:
from mlflow.genai.scorers import scorer

ROTULOS = {"DIFICULDADE": "dificuldade", "REGULAR": "regular", "BOM": "bom",
           "MUITO_BOM": "muito bom", "EXCELENTE": "excelente"}

def _texto(outputs): return (outputs or {}).get("recomendacao", "") if isinstance(outputs, dict) else str(outputs)

@scorer
def perfil_citado(outputs) -> bool:
    """A recomendação menciona o perfil previsto pelo classificador?"""
    return ROTULOS.get(outputs.get("perfil"), "?") in _texto(outputs).lower().replace("_", " ")

@scorer
def sem_pii(inputs, outputs) -> bool:
    """Nenhum nome ou matrícula do aluno na resposta."""
    aluno = inputs["aluno"]
    primeiro_nome = str(aluno.get("NOME", "")).split()[0].lower() if aluno.get("NOME") else "#"
    texto = _texto(outputs).lower()
    return primeiro_nome not in texto and str(aluno.get("MATRICULA", "#")) not in texto

@scorer
def tamanho_ok(outputs) -> bool:
    """No máximo 120 palavras (o coordenador lê dezenas por dia)."""
    return len(_texto(outputs).split()) <= 120

@scorer
def duas_acoes(outputs) -> bool:
    """Tem pelo menos 2 ações em lista (1. 2. ou marcadores)."""
    itens = re.findall(r"(?m)^\s*(?:\d+[.)]|[-*•])\s+", _texto(outputs))
    return len(itens) >= 2

SCORERS = [perfil_citado, sem_pii, tamanho_ok, duas_acoes]

In [ ]:
mlflow.openai.autolog()   # a avaliação também gera traces (um por aluno avaliado)

CAMPOS_PII = ["NOME", "MATRICULA"]

def gerador(versao_prompt):
    """Pipeline igual ao do serviço, mas com uma versão de prompt escolhida (avaliação offline)."""
    p = mlflow.genai.load_prompt(f"prompts:/mentoria/{versao_prompt}")
    def gerar(aluno):
        limpo = {k: v for k, v in aluno.items() if k not in CAMPOS_PII}
        perfil = requests.post(f"{CLASSIFICADOR_URL}/predict", timeout=10,
                               json={"fields": FEATURES, "values": [float(limpo[f]) for f in FEATURES]}).json()["_PERFIL_CALCULADO"]
        mensagens = p.format(perfil=perfil, dados=", ".join(f"{k}={limpo[k]}" for k in FEATURES))
        r = llm.chat.completions.create(model=LLM_MODELO, messages=mensagens, max_tokens=250, temperature=0.3,
                                        extra_body={"reasoning_effort": "none"})
        return {"perfil": perfil, "recomendacao": r.choices[0].message.content}
    return gerar

dados_avaliacao = [{"inputs": {"aluno": aluno_json(linha)}} for _, linha in alunos.iterrows()]

mlflow.set_experiment("Avaliacao Prompt Mentoria")
with mlflow.start_run(run_name="prompt v1"):
    res_v1 = mlflow.genai.evaluate(data=dados_avaliacao, predict_fn=gerador(1), scorers=SCORERS)   # ~2 min
res_v1.metrics

**No MLflow:** experimento `Avaliacao Prompt Mentoria` → run `prompt v1` → aba **Evaluations**: veja, aluno por aluno, qual scorer falhou e por quê.

Agora o prompt **v2**: regras explícitas de formato e um **dicionário dos dados**
(o modelo não sabe que `INGLES=-1` significa "sem resposta", e isso o leva a inventar).

In [ ]:
PROMPT_V2 = [
    {"role": "system", "content": (
        "Você é um coordenador pedagógico de uma faculdade e escreve em português do Brasil.\n"
        "Dicionário dos dados: REPROVACOES_MAT_n = reprovações na matéria de matemática n; "
        "NOTA_MAT_n = nota de 0 a 10; INGLES: 1 = fala inglês, 0 = não fala, -1 = sem resposta; "
        "H_AULA_PRES = horas de aula presencial por semana; TAREFAS_ONLINE = tarefas online entregues; FALTAS = faltas.\n"
        "Regras:\n"
        "1. Comece com 'Perfil: <perfil previsto>'.\n"
        "2. Em seguida, uma frase de diagnóstico usando somente os dados fornecidos (não invente).\n"
        "3. Depois, exatamente 2 ações de mentoria em lista numerada (1. e 2.).\n"
        "4. No máximo 80 palavras. Não mencione nome nem matrícula.")},
    {"role": "user", "content": "Perfil previsto: {{perfil}}\nDados do aluno: {{dados}}"},
]
v2 = mlflow.genai.register_prompt(name="mentoria", template=PROMPT_V2,
                                  commit_message="v2: dicionario de dados + regras de formato", tags={"autor": "aluno"})
print("Registrado mentoria versão", v2.version)

with mlflow.start_run(run_name="prompt v2"):
    res_v2 = mlflow.genai.evaluate(data=dados_avaliacao, predict_fn=gerador(v2.version), scorers=SCORERS)   # ~2 min

comparacao = pd.DataFrame({"v1": res_v1.metrics, "v2": res_v2.metrics})
comparacao

> **Responda:** o v2 melhorou em quais scorers? Piorou em algum? Que outro scorer você criaria?
>
> Se o v2 for melhor, **promova** para produção. O serviço `mentoria` passa a usá-lo em até 20 s, **sem rebuild nem restart**:

In [ ]:
mlflow.genai.set_prompt_alias("mentoria", alias="producao", version=v2.version)
time.sleep(21)
r = mentoria(aluno_json(alunos.iloc[2]), janela="parte_b")
print(r["prompt_versao"], "\n")
print(r["recomendacao"])

### (Opcional) LLM-as-a-judge

Um LLM avaliando o outro. Com um modelo de 1,7B em CPU cada julgamento leva **~45 s** e o juiz é pouco confiável:
serve para conhecer o conceito. Em produção, usa-se um modelo maior como juiz.

In [ ]:
from mlflow.genai.judges import make_judge

juiz = make_judge(
    name="adequacao",
    instructions="Avalie se a recomendação em {{ outputs }} é adequada e útil para o aluno descrito em {{ inputs }}. "
                 "Responda apenas 'sim' ou 'nao'.",
    feedback_value_type=str,
    model=f"openai:/{LLM_MODELO}",
)
amostra = {"aluno": {k: v for k, v in aluno_json(alunos.iloc[0]).items() if k not in CAMPOS_PII}}
saida = gerador(v2.version)(amostra["aluno"])
veredito = juiz(inputs=amostra, outputs=saida)
print("Veredito:", veredito.value, "\nJustificativa:", veredito.rationale)

---
# Parte C — Drift detection: o comportamento mudou?

O modelo de ML clássico sofre **drift** quando os dados mudam. Num sistema com LLM, há mais coisas que podem mudar:

| Tipo | O que muda | Como medir aqui |
|---|---|---|
| **Drift de entrada** | o perfil dos alunos que chegam | PSI da distribuição de perfis e das faltas |
| **Drift de saída** | o formato/tamanho/custo das respostas | tokens, latência e taxa de aprovação nos scorers |
| **Drift semântico** | o **conteúdo** das respostas | distância entre os *embeddings* das respostas |

Vamos gerar 3 janelas de tráfego (cada célula leva de 1 a 3 min):

In [ ]:
# Janela 1: tráfego normal (referência)
!python gerar_carga.py normal 8

In [ ]:
# Janela 2: a população mudou (só alunos com reprovação e muitas faltas)
!python gerar_carga.py drift-entrada 8

In [ ]:
# Janela 3: o sistema mudou (alguém trocou o modelo por um menor e subiu a temperatura)
!python gerar_carga.py drift-saida 8

In [ ]:
from scipy.stats import ks_2samp

df = traces_df("baseline", "drift_entrada", "drift_saida")
df["palavras"] = df.texto.str.split().str.len()
base = df[df.janela == "baseline"]

def psi(ref, atual, categorias):
    """Population Stability Index: < 0,1 estável | 0,1 a 0,2 atenção | > 0,2 drift."""
    r = ref.value_counts(normalize=True).reindex(categorias, fill_value=0) + 1e-4
    a = atual.value_counts(normalize=True).reindex(categorias, fill_value=0) + 1e-4
    return float(((a - r) * np.log(a / r)).sum())

def qualidade(linha):
    """Os mesmos critérios da Parte B, aplicados ao tráfego real (sem rótulo)."""
    saida = {"perfil": linha.perfil, "recomendacao": linha.texto}
    return np.mean([perfil_citado(outputs=saida), tamanho_ok(outputs=saida), duas_acoes(outputs=saida)])

df["qualidade"] = df.apply(qualidade, axis=1)
df.groupby("janela")[["latencia_s", "tokens_saida", "palavras", "qualidade"]].mean().round(2)

In [ ]:
def embeddings(textos):
    r = llm.embeddings.create(model=EMBED_MODELO, input=list(textos))
    v = np.array([d.embedding for d in r.data])
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def distancia_semantica(textos_ref, textos_atual):
    """1 - cosseno entre os centróides dos embeddings (0 = mesmo conteúdo médio)."""
    a, b = embeddings(textos_ref).mean(axis=0), embeddings(textos_atual).mean(axis=0)
    return float(1 - a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

# referência de ruído: metade da janela baseline contra a outra metade
metade = len(base) // 2
ruido = distancia_semantica(base.texto[:metade], base.texto[metade:])
print(f"Distância semântica baseline x baseline (ruído natural): {ruido:.4f}")

In [ ]:
LIMITES = {"psi_perfil": 0.2, "qualidade_min": 0.7, "variacao_tokens": 0.5, "semantica_fator": 3}
PERFIS = list(ROTULOS)

mlflow.set_experiment("Monitoramento LLM")
resultado = []
for janela in ["drift_entrada", "drift_saida"]:
    atual = df[df.janela == janela]
    m = {
        "psi_perfil": psi(base.perfil, atual.perfil, PERFIS),
        "ks_faltas_pvalor": float(ks_2samp(base.FALTAS, atual.FALTAS).pvalue),
        "variacao_tokens": float(atual.tokens_saida.mean() / base.tokens_saida.mean() - 1),
        "variacao_latencia": float(atual.latencia_s.mean() / base.latencia_s.mean() - 1),
        "qualidade": float(atual.qualidade.mean()),
        "dist_semantica": distancia_semantica(base.texto, atual.texto),
    }
    alertas = []
    if m["psi_perfil"] > LIMITES["psi_perfil"]: alertas.append("drift de entrada (perfis)")
    if m["qualidade"] < LIMITES["qualidade_min"]: alertas.append("queda de qualidade")
    if abs(m["variacao_tokens"]) > LIMITES["variacao_tokens"]: alertas.append("drift de saída (tokens)")
    if m["dist_semantica"] > LIMITES["semantica_fator"] * ruido: alertas.append("drift semântico")

    with mlflow.start_run(run_name=f"baseline x {janela}"):
        mlflow.log_metrics(m)
        mlflow.log_metric("dist_semantica_ruido", ruido)
        mlflow.set_tags({"janela": janela, "alertas": ", ".join(alertas) or "nenhum",
                         "modelo": ",".join(atual.modelo.unique())})
    resultado.append({"janela": janela, **{k: round(v, 3) for k, v in m.items()}, "ALERTAS": alertas or "-"})

pd.DataFrame(resultado).set_index("janela").T

In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
pd.crosstab(df.janela, df.perfil, normalize="index").plot.bar(stacked=True, ax=eixos[0], title="Perfis por janela (entrada)")
df.groupby("janela").tokens_saida.mean().plot.bar(ax=eixos[1], title="Tokens de saída (média)")
df.groupby("janela").qualidade.mean().plot.bar(ax=eixos[2], title="Qualidade (scorers)", ylim=(0, 1))
for e in eixos: e.tick_params(axis="x", rotation=0)
plt.tight_layout()

**No MLflow:** experimento `Monitoramento LLM` → selecione os 2 runs → **Compare**: as métricas de cada janela ficam lado a lado, com a tag `alertas`.

> **Perguntas finais da Parte 2:**
> 1. Na janela `drift_entrada`, o LLM piorou? Ou foi a população de alunos que mudou? Qual métrica mostra cada coisa?
> 2. Na janela `drift_saida`, ninguém mexeu nos dados. O que denunciou a troca de modelo?
> 3. Por que comparamos a distância semântica com o "ruído natural" da própria baseline em vez de usar um valor fixo?
> 4. Quais desses alertas você mandaria para o time de plantão? Quais iriam só para um relatório semanal?
> 5. Em produção, de onde viriam a janela de referência e o gatilho para rodar este monitoramento?